In [ ]:

import pandas as pd

# 读取训练集和测试集
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 预览数据
print("训练集前5行：")
print(train_df.head())
print("\n测试集前5行：")
print(test_df.head())


训练集前5行：
      id  Gender  ...                 MTRANS           NObeyesdad
0   9958    Male  ...             Automobile       Obesity_Type_I
1   7841    Male  ...  Public_Transportation  Insufficient_Weight
2   9293    Male  ...  Public_Transportation      Obesity_Type_II
3  15209  Female  ...             Automobile       Obesity_Type_I
4  16515    Male  ...  Public_Transportation  Overweight_Level_II

[5 rows x 18 columns]

测试集前5行：
      id  Gender  ...                 MTRANS           NObeyesdad
0  10317  Female  ...  Public_Transportation     Obesity_Type_III
1   4074    Male  ...  Public_Transportation   Overweight_Level_I
2   9060  Female  ...  Public_Transportation       Obesity_Type_I
3  11286    Male  ...  Public_Transportation      Obesity_Type_II
4   8254    Male  ...  Public_Transportation  Insufficient_Weight

[5 rows x 18 columns]


In [ ]:


# 检查数据结构和缺失值
print("训练集数据结构：")
print(train_df.info())
print("\n训练集缺失值情况：")
print(train_df.isnull().sum())

print("\n测试集数据结构：")
print(test_df.info())
print("\n测试集缺失值情况：")
print(test_df.isnull().sum())



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

), int64(1), object(9)
memory usage: 2.3+ MB
None

训练集缺失值情况：
id                                0
Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64

测试集数据结构：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4152 entries, 0 to 4151
Data columns (total 18 colum

In [ ]:


import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 重新读取数据
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 特征构建
def add_features(df):
    # 计算 BMI
    df['BMI'] = df['Weight'] / (df['Height'] / 100) ** 2
    # 创建新的特征，如是否经常运动
    df['Frequent_Exercise'] = (df['FAF'] > 1) & (df['TUE'] > 1)
    return df

train_df = add_features(train_df)
test_df = add_features(test_df)

# 特征选择
features = ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'BMI', 'Frequent_Exercise']
target = 'NObeyesdad'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# 数据预处理
numeric_features = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'BMI']
categorical_features = ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS', 'Frequent_Exercise']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 创建模型管道
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 训练模型
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

# 评估模型
accuracy = accuracy_score(y_test, y_pred)
print(f"模型在测试集上的准确率: {accuracy:.4f}")



模型在测试集上的准确率: 0.9003
